In [ ]:
#| default_exp loss

# Loss functions

> Im lost too. 

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import torch, numpy as np, torch.nn.functional as F, torch.nn as nn


In [ ]:
#| export
def mse_loss(preds, target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.mse_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.mse_loss(preds, target, reduction='mean')

def mae_loss(preds, target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.l1_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.l1_loss(preds, target, reduction='mean')


def mape(preds, target):
    epsilon = np.finfo(np.float64).eps # from sklearn
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += (((targ - pred).mean(dim=-1).abs()) / (targ.mean(dim=-1).abs()).clamp(min=epsilon)).sum()
        return loss / preds.size(0)
    else:
        return (((target - preds).mean(dim=-1).abs()) / (target.mean(dim=-1).abs()).clƒamp(min=epsilon)).sum()
    

def cosine_similarity_loss(preds,target):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            sim = F.cosine_similarity(pred, targ, dim=-1)
            loss += sim.sum() / torch.as_tensor(torch.numel(pred), device=preds.device)
        return -loss / preds.size(0)
    else:
        sim = F.cosine_similarity(preds,target, dim=-1)
        n_elements = torch.numel(preds)
        sim = sim.sum() / torch.as_tensor(n_elements, device=preds.device)
        return -sim

def huber_loss(preds, target, delta=1):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    """
    if preds.is_nested:
        # Calculate loss for each item in the batch separately
        total_loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            # Calculate huber loss with reduction='none' for this item
            total_loss += F.huber_loss(pred, targ, delta=delta, reduction='mean')
            
        # Calculate average loss across all actual elements
        return total_loss / preds.size(0)
    else:   
        return F.huber_loss(preds, target, delta=delta, reduction='mean')
    

def smoothl1_loss(preds, target):
    if preds.is_nested:
        loss = 0
        for pred, targ in zip(preds.unbind(), target.unbind()):
            loss += F.smooth_l1_loss(pred, targ, reduction='mean')
        return loss / preds.size(0)
    else:
        return F.smooth_l1_loss(preds, target, reduction='mean')

def mse_variance_loss(preds, target, representations, alpha = 0.2):
    """
    preds:   [bs x num_patch x n_vars x patch_len]
    targets: [bs x num_patch x n_vars x patch_len] 
    representations: [bs x nvars x d_model x num_patch]
    """
    loss = 0.0
    for pred_i, target_i, representations_i in zip(preds, target, representations):
        loss = loss + F.mse_loss(pred_i, target_i, reduction='mean')
        loss = loss + alpha * F.relu(1.0 - representations_i.std(dim=-1)).mean()
    loss /= len(preds)
    return loss


In [ ]:
#| export
def nll_logistic_hazard(phi, events, idx_durations, reduction = 'mean'):
    """
    Adapted from https://github.com/havakv/pycox/blob/3eccdd7fd9844a060f50fdcc315659f33a2d2dc1/pycox/models/loss.py#L18
    Negative log-likelihood of the discrete time hazard parametrized model LogisticHazard [1].
    
    Arguments:
        phi {torch.tensor} -- Estimates in (-inf, inf), where hazard = sigmoid(phi).
        idx_durations {torch.tensor} -- Event times represented as indices.
        events {torch.tensor} -- Indicator of event (1.) or censoring (0.).
            Same length as 'idx_durations'.
        reduction {string} -- How to reduce the loss.
            'none': No reduction.
            'mean': Mean of tensor.
            'sum: sum.
    
    Returns:
        torch.tensor -- The negative log-likelihood.

    References:
    [1] Håvard Kvamme and Ørnulf Borgan. Continuous and Discrete-Time Survival Prediction
        with Neural Networks. arXiv preprint arXiv:1910.06724, 2019.
        https://arxiv.org/pdf/1910.06724.pdf
    """
    if phi.shape[-1] <= idx_durations.max():
        raise ValueError(f"Network output `phi` is too small for `idx_durations`."+
                         f" Need at least `phi.shape[1] = {idx_durations.max().item()+1}`,"+
                         f" but got `phi.shape[1] = {phi.shape[1]}`")
    phi = phi.float()
    events = events.float()

    if events.dim() == 1:
        events = events.view(-1, 1)
        idx_durations = idx_durations.view(-1, 1)
        y_bce = torch.zeros_like(phi).scatter(1, idx_durations, events)
        bce = F.binary_cross_entropy_with_logits(phi, y_bce, reduction='none')
        loss = bce.cumsum(1).gather(1, idx_durations).view(-1)
    elif events.dim() == 2:
        events_1 = events[:, 0].view(-1, 1)
        idx_durations_1 = idx_durations[:, 0].view(-1, 1)
        y_bce_1 = torch.zeros_like(phi).scatter(1, idx_durations_1, events_1)
        bce_1 = F.binary_cross_entropy_with_logits(phi, y_bce_1, reduction='none')
        loss_1 = bce_1.cumsum(1).gather(1, idx_durations_1).view(-1)

        events_2 = events[:, 1].view(-1, 1)
        idx_durations_2 = idx_durations[:, 1].view(-1, 1)
        y_bce_2 = torch.zeros_like(phi).scatter(1, idx_durations_2, events_2)
        bce_2 = F.binary_cross_entropy_with_logits(phi, y_bce_2, reduction='none')
        loss_2 = bce_2.cumsum(1).gather(1, idx_durations_2).view(-1)

        loss = loss_1 + loss_2

    if reduction == 'none':
        return loss
    elif reduction == 'mean':
        return loss.mean()
    elif reduction == 'sum':
        return loss.sum()
    else:
        raise ValueError(f"Reduction {reduction} not implemented.")

In [ ]:
#| export
class CrossEntropyLoss(nn.Module):
    """
    Cross entropy loss with ignore_index.
    """
    def __init__(self, ignore_index=-100, reduction='mean', weight=None, label_smoothing=0, soft_labels=False):
        super().__init__()
        self.ignore_index = ignore_index
        self.reduction = reduction
        self.label_smoothing = label_smoothing
        self.weight = weight
        if weight is not None:
            self.weight = weight.float()
        if soft_labels:
            # this has to be -100 for CE to work with soft labels
            ignore_index = -100
        self.soft_labels = soft_labels
        self.loss = nn.CrossEntropyLoss(ignore_index=ignore_index, reduction='none', weight=self.weight, label_smoothing=label_smoothing)
    
    def forward(self, x, y):
        """
        x: [bs x n classes x n patches]
        y: [bs x n patches] or [bs x n classes x n patches]
        """
        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.ignore_index)
        loss = self.loss(x, y)
        if self.soft_labels and y.dim() == 3:
            mask = y.sum(dim=1, keepdim=True) > 0  # [bs x 1 x soft_labels]
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-6) if self.reduction == 'mean' else loss.sum()
        else:
            mask = y != self.ignore_index
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-6) if self.reduction == 'mean' else loss.sum()
        return loss


In [ ]:
#| export
class FocalLoss(nn.Module):
    """
    adapted from tsai, weighted multiclass focal loss
    https://github.com/timeseriesAI/tsai/blob/bdff96cc8c4c8ea55bc20d7cffd6a72e402f4cb2/tsai/losses.py#L116C1-L140C20
    """
    def __init__(self, 
                 weight=None, 
                 gamma=2., 
                 reduction='mean',
                 ignore_index=-100
                 ):
        super().__init__()
        self.weight = weight
        self.gamma = gamma
        self.reduction = reduction
        self.ignore_index = ignore_index
    
    __name__ = 'focalloss'
        
    def forward(self, x, y):
        """
        x: [bs x n classes x n patches]
        y: [bs x n patches]
        """
        bs = x.size(0)

        if y.dtype == torch.float32:
            y = y.long()

        if x.is_nested:
            x = x.to_padded_tensor(padding=0)
        if y.is_nested:
            y = y.to_padded_tensor(padding=self.ignore_index)

        if x.dim() == 3:
            x = x.permute(0,2,1) # bs x n_patches x n_classes
            x = x.reshape(-1, x.size(-1)) # bs * n_patches x n_classes

        log_prob = F.log_softmax(x, dim=-1)
        weight = self.weight.float().to(x.device) if self.weight is not None else None
        if y.dim() == 2:
            # hard labels
            y = y.flatten(start_dim=-2) # bs * num_patches
            valid_mask = y != self.ignore_index
            valid_y = y[valid_mask]
            valid_log_prob = log_prob[valid_mask]
            pt = valid_log_prob[torch.arange(len(valid_y)), valid_y].exp()
            ce = F.nll_loss(valid_log_prob, valid_y, weight=weight, reduction='none', ignore_index=self.ignore_index)
            loss = (1 - pt) ** self.gamma * ce
            loss = loss.sum() / valid_mask.sum().clamp(min=1e-5) if self.reduction == 'mean' else loss.sum()
        else:  # soft labels
            y = y.permute(0,2,1) # bs x n patches x n classes
            mask = (y.sum(dim=2) > 0).float() # bs x n patches
            mask = mask.flatten(start_dim=-2) # bs * n_patches
            y = y.reshape(-1, y.size(-1)) # bs * n_patches x n_classes

            if weight is not None:
                ce = -(y * log_prob * weight).sum(dim=-1) # bs * n_patches
            else:
                ce = -(y * log_prob).sum(dim=-1) # bs * n_patches
            pt = (y * log_prob.exp()).sum(dim=-1).clamp(min=1e-7, max=1)
            loss = (1 - pt) ** self.gamma * ce
            # Positions to ignore will have all zeros
            loss = loss * mask
            loss = loss.sum() / mask.sum().clamp(min=1e-5) if self.reduction == 'mean' else loss.sum()
        return loss

In [ ]:
#| export
class KLDivLoss(nn.Module):
    """
    Kullback-Leibler Divergence Loss with masking for ignore_index.
    Handles soft labels with ignore_index marked as -100.
    
    Args:
        logits: [bs x n_classes x pred_labels] - model predictions
        targets: [bs x n_classes x soft_labels] - soft labels, with ignore_index positions marked as 0
    """
    def __init__(self, reduction='mean'):
        super().__init__()
        self.kl_loss = nn.KLDivLoss(reduction='none')
        self.reduction = reduction
    def forward(self, logits, targets):
        bs = logits.size(0)
        if logits.is_nested:
            max_len = max(logits.size(-1) for logits in logits.unbind())
            logits = logits.to_padded_tensor(padding=0, output_size=(bs, logits.size(1), max_len))
            targets = targets.to_padded_tensor(padding=0, output_size=(bs, targets.size(1), max_len))
        # Create mask for valid positions (where target is not ignore_index)
        mask = targets.sum(dim=1, keepdim=True) > 0  # [bs x 1 x soft_labels]
        # Compute log probabilities
        log_probs = F.log_softmax(logits, dim=1)
        # Compute KL divergence loss
        loss = self.kl_loss(log_probs, targets)
        # Sum across class dimension and apply mask
        loss = (loss * mask).sum() / mask.sum().clamp(min=1e-6) if self.reduction == 'mean' else (loss * mask).sum()
        
        return loss

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()